# Bloc 4 — Monte Carlo, Markov Chains, MCMC & Copulas
## Simulation probabiliste et dépendance temporelle

## Étape 1 — Monte Carlo Scenario Generation
### Génération de scénarios de consommation

### Partie A — Mise en pratique

In [17]:
# ============================================================
# BLOCK 4 — INITIALISATION
# Recharge les données et résultats des blocs précédents
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Données principales
# ------------------------------------------------------------

X = joblib.load(
    "data_processed/X_block1_final.joblib"
)

y = joblib.load(
    "data_processed/y_block1_final.joblib"
)

X_weather_plus = joblib.load(
    "data_processed/X_block3_weather_nonlinear.joblib"
)

y_weather = joblib.load(
    "data_processed/y_block3_weather.joblib"
)

# ------------------------------------------------------------
# 2. Modèle final de régression utilisé au Bloc 3
# ------------------------------------------------------------

conformal_model = joblib.load(
    "models/block3_conformal_ridge.joblib"
)

# ------------------------------------------------------------
# 3. Découpage temporel
# ------------------------------------------------------------

train = (
    X_weather_plus.index < "2024-01-01"
)

cal = (
    (X_weather_plus.index >= "2024-01-01")
    & (X_weather_plus.index < "2025-01-01")
)

val = (
    (X_weather_plus.index >= "2025-01-01")
    & (X_weather_plus.index < "2026-01-01")
)

# ------------------------------------------------------------
# 4. Prédictions 2024 et 2025
# ------------------------------------------------------------

pred_cal = conformal_model.predict(
    X_weather_plus.loc[cal]
)

pred_val = conformal_model.predict(
    X_weather_plus.loc[val]
)

# ------------------------------------------------------------
# 5. Valeurs réelles
# ------------------------------------------------------------

y_cal = y_weather.loc[cal].values
y_val = y_weather.loc[val].values

cal_index = X_weather_plus.loc[cal].index
val_index = X_weather_plus.loc[val].index

# ------------------------------------------------------------
# 6. Résidus de calibration 2024
# ------------------------------------------------------------

residual_cal = (
    y_cal - pred_cal
)

scores_cal = np.abs(
    residual_cal
)

# ------------------------------------------------------------
# 7. Résultats EVT du Bloc 3
# ------------------------------------------------------------

Q99 = 87004.241599
Q995 = 89093.314260
Q999 = 93040.056204

EVT_THRESHOLD = 81005.0

# ------------------------------------------------------------
# 8. Vérification
# ------------------------------------------------------------

print("X Block 1 :", X.shape)
print("X météo   :", X_weather_plus.shape)

print("\nCalibration 2024 :", cal.sum())
print("Validation 2025  :", val.sum())

print("\nRésidu moyen :", residual_cal.mean())
print("Écart-type résidus :", residual_cal.std())

print("\nQ99   :", Q99)
print("Q99.5 :", Q995)
print("Q99.9 :", Q999)

print("\n✅ Initialisation Bloc 4 terminée")

X Block 1 : (236254, 416)
X météo   : (227567, 429)

Calibration 2024 : 17568
Validation 2025  : 17517

Résidu moyen : -0.32927285936314893
Écart-type résidus : 207.84024761268705

Q99   : 87004.241599
Q99.5 : 89093.31426
Q99.9 : 93040.056204

✅ Initialisation Bloc 4 terminée


In [18]:
block3_state = {
    "Q99": Q99,
    "Q995": Q995,
    "Q999": Q999,
    "EVT_THRESHOLD": EVT_THRESHOLD,
    "calibration_start": "2024-01-01",
    "validation_start": "2025-01-01",
    "test_start": "2026-01-01"
}

joblib.dump(
    block3_state,
    "models/block3_state.joblib"
)

['models/block3_state.joblib']

In [19]:
residual_cal = (
    y_weather.loc[cal].values
    - pred_cal
)

print("Mean residual :", residual_cal.mean())
print("Std residual :", residual_cal.std())

Mean residual : -0.32927285936314893
Std residual : 207.84024761268705


In [20]:
val_index = X_weather_plus.loc[val].index

pred_series = pd.Series(
    pred_val,
    index=val_index
)

daily_pred_peak = (
    pred_series
    .resample("D")
    .max()
)

selected_day = daily_pred_peak.idxmax()

print("Selected day :", selected_day)

Selected day : 2025-01-14 00:00:00+00:00


In [21]:
day_mask = (
    val_index.normalize()
    == selected_day.normalize()
)

pred_day = pred_val[day_mask]
y_day = y_val[day_mask]

print("Number of points :", len(pred_day))
print("Predicted peak :", pred_day.max())
print("Observed peak :", y_day.max())

Number of points : 48
Predicted peak : 87871.40957420101
Observed peak : 87716.0


In [22]:
rng = np.random.default_rng(42)

N_SIM = 5000

simulated_residuals = rng.choice(
    residual_cal,
    size=(N_SIM, len(pred_day)),
    replace=True
)

scenarios_mc = (
    pred_day[None, :]
    + simulated_residuals
)

In [23]:
evt_threshold = 81005.0

scenario_peaks = scenarios_mc.max(axis=1)

prob_extreme = np.mean(
    scenario_peaks > evt_threshold
)

print(
    "Mean simulated peak :",
    scenario_peaks.mean()
)

print(
    "95% peak quantile :",
    np.quantile(scenario_peaks, 0.95)
)

print(
    "P(peak > 81 005 MW) :",
    prob_extreme
)

Mean simulated peak : 87970.45961491745
95% peak quantile : 88244.90257013637
P(peak > 81 005 MW) : 1.0


In [24]:
mc_results = {
    "selected_day": selected_day,
    "n_simulations": N_SIM,
    "mean_peak": scenario_peaks.mean(),
    "q95_peak": np.quantile(
        scenario_peaks, 0.95
    ),
    "prob_peak_above_evt": prob_extreme
}

joblib.dump(
    mc_results,
    "models/block4_mc_baseline.joblib"
)

['models/block4_mc_baseline.joblib']

### Partie C — Analyse

Le Monte Carlo permet de transformer une prévision ponctuelle en plusieurs scénarios possibles :

$$
Y_t^{(s)}
=
\hat Y_t+\varepsilon_t^{(s)}
$$

La première méthode tire les résidus indépendamment dans la distribution empirique observée en 2024.

Cette approche reproduit correctement la distribution marginale des erreurs, mais elle ignore leur dépendance temporelle.

Deux erreurs consécutives dans une série électrique ne sont généralement pas indépendantes.

La prochaine étape vérifiera cette limite puis utilisera une copule afin de générer des scénarios temporellement plus réalistes.

In [25]:
thresholds_evt = {
    "Q99": 87004.24,
    "Q995": 89093.31,
    "Q999": 93040.06
}

for name, threshold in thresholds_evt.items():

    prob = np.mean(
        scenario_peaks > threshold
    )

    print(
        name,
        ":", prob
    )

Q99 : 1.0
Q995 : 0.0016
Q999 : 0.0


#### Conclusion sur le risque de pic

Pour la journée sélectionnée, le dépassement de $Q_{0.99}$ est pratiquement certain, car le pic prévu est déjà supérieur à ce seuil.

En revanche, le dépassement de niveaux plus extrêmes reste très rare :

$$
P(Peak > Q_{0.995}) \approx 0.16\%
$$

et aucun dépassement de $Q_{0.999}$ n'est observé parmi les 5 000 scénarios.

Le Monte Carlo permet donc de transformer une prévision ponctuelle en estimation probabiliste du risque de dépassement.

### Conclusion sur la dépendance temporelle

Les résidus présentent une dépendance temporelle modérée.

L'autocorrélation est d'environ :

$$
0.10
$$

à 30 minutes et atteint :

$$
0.18
$$

à une semaine.

La persistance du signe est proche de 50 %, ce qui indique une faible dépendance directionnelle à très court terme.

Le Monte Carlo indépendant constitue donc une approximation raisonnable, mais il ne reproduit pas toute la structure temporelle des erreurs.

On teste maintenant une copule afin de générer des trajectoires journalières dépendantes.

## Étape 2 — Temporal Dependence
### Dépendance temporelle des erreurs

### Partie A — Mise en pratique

Le Monte Carlo précédent suppose que les erreurs sont indépendantes.

On vérifie maintenant si cette hypothèse est réaliste.

In [26]:
resid_series = pd.Series(
    residual_cal,
    index=cal_index
)

lags = {
    "30 min": 1,
    "1 h": 2,
    "1 day": 48,
    "1 week": 336
}

acf_results = pd.DataFrame({
    "Lag": lags.keys(),
    "Autocorrelation": [
        resid_series.autocorr(lag=k)
        for k in lags.values()
    ]
})

print(acf_results)

      Lag  Autocorrelation
0  30 min         0.101409
1     1 h         0.046703
2   1 day         0.040205
3  1 week         0.175109


In [27]:
same_sign = np.mean(
    np.sign(residual_cal[1:])
    ==
    np.sign(residual_cal[:-1])
)

print(
    "Same sign consecutive residuals :",
    same_sign
)

Same sign consecutive residuals : 0.5075425513747367


### Partie C — Analyse

Les résidus présentent une dépendance temporelle modérée.

À court terme :

$$
\rho_{30min}\approx0.10
$$

et :

$$
\rho_{1h}\approx0.05
$$

La dépendance journalière est faible :

$$
\rho_{1day}\approx0.04
$$

mais une structure hebdomadaire plus nette apparaît :

$$
\rho_{1week}\approx0.18
$$

La probabilité que deux résidus consécutifs aient le même signe est d'environ :

$$
50.75\%
$$

ce qui indique une faible persistance directionnelle immédiate.

Le Monte Carlo indépendant constitue donc une approximation raisonnable, mais il ne reproduit pas toute la dépendance temporelle.

La prochaine étape consiste à simuler des journées complètes en préservant la dépendance entre les 48 demi-heures.

## Étape 3 — Gaussian Copula
### Simulation de scénarios avec dépendance temporelle

### Partie A — Mise en pratique

Au lieu de tirer indépendamment les 48 erreurs d'une journée, on modélise conjointement :

$$
(\varepsilon_1,\ldots,\varepsilon_{48})
$$

afin de préserver la dépendance entre les différentes demi-heures.

In [28]:
resid_df = pd.DataFrame({
    "residual": residual_cal
}, index=cal_index)

resid_df["date"] = resid_df.index.date
resid_df["slot"] = (
    resid_df.index.hour * 2
    + resid_df.index.minute // 30
)

daily_residuals = (
    resid_df
    .pivot(
        index="date",
        columns="slot",
        values="residual"
    )
    .dropna()
)

print("Jours complets :", daily_residuals.shape[0])
print("Variables par jour :", daily_residuals.shape[1])

Jours complets : 366
Variables par jour : 48


In [29]:
from scipy.stats import norm

R = daily_residuals.to_numpy()

n_days, n_slots = R.shape

U = np.empty_like(R, dtype=float)

for j in range(n_slots):
    U[:, j] = (
        pd.Series(R[:, j])
        .rank(method="average")
        .to_numpy()
        - 0.5
    ) / n_days

Z = norm.ppf(U)

copula_corr = np.corrcoef(
    Z,
    rowvar=False
)

joblib.dump(
    copula_corr,
    "models/block4_gaussian_copula_corr.joblib"
)

print("Matrice :", copula_corr.shape)

Matrice : (48, 48)


In [30]:
pred_series = pd.Series(
    pred_val,
    index=val_index
)

selected_day = (
    pred_series
    .resample("D")
    .max()
    .idxmax()
)

day_mask = (
    val_index.normalize()
    == selected_day.normalize()
)

pred_day = pred_val[day_mask]

print("Jour :", selected_day)
print("Points :", len(pred_day))

Jour : 2025-01-14 00:00:00+00:00
Points : 48


In [31]:
rng = np.random.default_rng(42)

N_SIM = 5000

Z_sim = rng.multivariate_normal(
    np.zeros(48),
    copula_corr,
    size=N_SIM
)

U_sim = norm.cdf(Z_sim)

residuals_copula = np.empty_like(U_sim)

for j in range(48):
    residuals_copula[:, j] = np.quantile(
        R[:, j],
        U_sim[:, j]
    )

scenarios_copula = (
    pred_day[None, :]
    + residuals_copula
)

peaks_copula = scenarios_copula.max(axis=1)

In [32]:
iid_residuals = rng.choice(
    residual_cal,
    size=(N_SIM, 48),
    replace=True
)

scenarios_iid = (
    pred_day[None, :]
    + iid_residuals
)

peaks_iid = scenarios_iid.max(axis=1)

In [33]:
comparison_sim = pd.DataFrame({
    "Method": [
        "Monte Carlo IID",
        "Gaussian Copula"
    ],

    "Mean_peak": [
        peaks_iid.mean(),
        peaks_copula.mean()
    ],

    "Q95_peak": [
        np.quantile(peaks_iid, 0.95),
        np.quantile(peaks_copula, 0.95)
    ],

    "P_Q995": [
        np.mean(peaks_iid > Q995),
        np.mean(peaks_copula > Q995)
    ],

    "P_Q999": [
        np.mean(peaks_iid > Q999),
        np.mean(peaks_copula > Q999)
    ]
})

print(comparison_sim)

joblib.dump(
    comparison_sim,
    "models/block4_copula_comparison.joblib"
)

            Method     Mean_peak      Q95_peak  P_Q995  P_Q999
0  Monte Carlo IID  87975.938107  88274.989862  0.0014     0.0
1  Gaussian Copula  87950.325124  88214.861859  0.0000     0.0


['models/block4_copula_comparison.joblib']

### Partie C — Première analyse

La copule gaussienne produit des pics légèrement moins élevés que le Monte Carlo IID.

Cependant, cette comparaison mélange deux effets :

- la dépendance temporelle ;
- les distributions différentes des erreurs selon la demi-heure.

Pour mesurer uniquement l'effet de la dépendance, on compare maintenant deux simulations ayant exactement les mêmes distributions marginales par demi-heure.

In [34]:
rng = np.random.default_rng(42)

N_SIM = 20000

# =====================================
# 1. Marginales par slot, indépendantes
# =====================================

U_ind = rng.random(
    (N_SIM, 48)
)

resid_ind = np.empty_like(U_ind)

for j in range(48):
    resid_ind[:, j] = np.quantile(
        R[:, j],
        U_ind[:, j]
    )

peaks_ind = (
    pred_day[None, :]
    + resid_ind
).max(axis=1)


# =====================================
# 2. Même marginales + Copule
# =====================================

Z_sim = rng.multivariate_normal(
    np.zeros(48),
    copula_corr,
    size=N_SIM
)

U_cop = norm.cdf(Z_sim)

resid_cop = np.empty_like(U_cop)

for j in range(48):
    resid_cop[:, j] = np.quantile(
        R[:, j],
        U_cop[:, j]
    )

peaks_cop = (
    pred_day[None, :]
    + resid_cop
).max(axis=1)

In [35]:
comparison_dependence = pd.DataFrame({
    "Method": [
        "Independent slotwise",
        "Gaussian Copula"
    ],
    "Mean_peak": [
        peaks_ind.mean(),
        peaks_cop.mean()
    ],
    "Q95_peak": [
        np.quantile(peaks_ind, 0.95),
        np.quantile(peaks_cop, 0.95)
    ],
    "Q99_peak": [
        np.quantile(peaks_ind, 0.99),
        np.quantile(peaks_cop, 0.99)
    ],
    "P_Q995": [
        np.mean(peaks_ind > Q995),
        np.mean(peaks_cop > Q995)
    ]
})

print(comparison_dependence)

joblib.dump(
    comparison_dependence,
    "models/block4_dependence_comparison.joblib"
)

                 Method     Mean_peak      Q95_peak      Q99_peak  P_Q995
0  Independent slotwise  87965.833218  88232.408209  88375.813697     0.0
1       Gaussian Copula  87954.194360  88226.722103  88374.201525     0.0


['models/block4_dependence_comparison.joblib']

### Conclusion de l'Étape 3

Après avoir contrôlé les distributions marginales par demi-heure, la copule gaussienne modifie très peu la distribution des pics.

Le quantile 99 % passe seulement de :

$$
88375.8
$$

à :

$$
88374.2
$$

MW.

La dépendance intra-journalière a donc peu d'influence sur le risque de pic dans cette expérience.

La copule reste intéressante pour générer des trajectoires temporelles cohérentes, mais elle n'apporte pas d'amélioration notable pour l'estimation du pic journalier.

## Étape 4 — Markov Chains
### Régimes d'erreur et transitions temporelles

### Partie A — Mise en pratique

On discrétise les résidus en trois états :

- erreur faible ;
- erreur normale ;
- erreur élevée.

Une chaîne de Markov permet ensuite d'étudier la probabilité de passer d'un régime à un autre.

In [36]:
q20, q80 = np.quantile(
    residual_cal,
    [0.20, 0.80]
)

states = np.where(
    residual_cal < q20,
    0,
    np.where(
        residual_cal > q80,
        2,
        1
    )
)

print("q20 :", q20)
print("q80 :", q80)

print(
    pd.Series(states)
    .value_counts(normalize=True)
    .sort_index()
)

q20 : -141.4640541486631
q80 : 141.39105802734707
0    0.200023
1    0.599954
2    0.200023
Name: proportion, dtype: float64


In [37]:
transition_counts = np.zeros(
    (3, 3),
    dtype=int
)

for a, b in zip(
    states[:-1],
    states[1:]
):
    transition_counts[a, b] += 1

transition_matrix = (
    transition_counts
    / transition_counts.sum(
        axis=1,
        keepdims=True
    )
)

transition_df = pd.DataFrame(
    transition_matrix,
    index=[
        "Low error",
        "Normal",
        "High error"
    ],
    columns=[
        "Low error",
        "Normal",
        "High error"
    ]
)

print(transition_df.round(3))

            Low error  Normal  High error
Low error       0.247   0.547       0.206
Normal          0.186   0.631       0.183
High error      0.195   0.560       0.245


In [38]:
print(
    "Persistance Low    :",
    transition_matrix[0, 0]
)

print(
    "Persistance Normal :",
    transition_matrix[1, 1]
)

print(
    "Persistance High   :",
    transition_matrix[2, 2]
)

Persistance Low    : 0.24672737620944793
Persistance Normal : 0.6310246679316888
Persistance High   : 0.24537432393965272


### Partie C — Analyse

La chaîne de Markov révèle une légère persistance des régimes d'erreur.

Les états extrêmes représentent chacun environ :

$$
20\%
$$

des observations, mais leur probabilité de rester dans le même état atteint environ :

$$
24.5\%
$$

L'état normal représente environ :

$$
60\%
$$

des observations et présente une persistance d'environ :

$$
63.1\%
$$

La dépendance entre régimes existe donc, mais reste modérée.

Une chaîne de Markov peut néanmoins produire des séquences d'erreurs plus réalistes qu'un tirage totalement indépendant.

## Étape 5 — Markov Scenario Simulation
### Simulation de scénarios par chaîne de Markov

### Partie A — Mise en pratique

In [39]:
residuals_by_state = {
    0: residual_cal[states == 0],
    1: residual_cal[states == 1],
    2: residual_cal[states == 2]
}

In [40]:
rng = np.random.default_rng(42)

def simulate_markov_day():

    simulated_states = np.empty(
        48,
        dtype=int
    )

    simulated_residuals = np.empty(48)

    # État initial selon les fréquences marginales
    simulated_states[0] = rng.choice(
        [0, 1, 2],
        p=[0.2, 0.6, 0.2]
    )

    simulated_residuals[0] = rng.choice(
        residuals_by_state[
            simulated_states[0]
        ]
    )

    for t in range(1, 48):

        previous = simulated_states[t - 1]

        simulated_states[t] = rng.choice(
            [0, 1, 2],
            p=transition_matrix[previous]
        )

        simulated_residuals[t] = rng.choice(
            residuals_by_state[
                simulated_states[t]
            ]
        )

    return simulated_states, simulated_residuals

In [41]:
N_SIM = 5000

markov_residuals = np.empty(
    (N_SIM, 48)
)

for i in range(N_SIM):

    _, r = simulate_markov_day()

    markov_residuals[i] = r

In [42]:
scenarios_markov = (
    pred_day[None, :]
    + markov_residuals
)

peaks_markov = scenarios_markov.max(
    axis=1
)

In [43]:
comparison_markov = pd.DataFrame({
    "Method": [
        "Independent",
        "Markov"
    ],

    "Mean_peak": [
        peaks_ind.mean(),
        peaks_markov.mean()
    ],

    "Q95_peak": [
        np.quantile(peaks_ind, 0.95),
        np.quantile(peaks_markov, 0.95)
    ],

    "Q99_peak": [
        np.quantile(peaks_ind, 0.99),
        np.quantile(peaks_markov, 0.99)
    ]
})

print(comparison_markov)

        Method     Mean_peak      Q95_peak      Q99_peak
0  Independent  87965.833218  88232.408209  88375.813697
1       Markov  87977.106750  88288.943143  88565.704991


In [44]:
sim_states = []

for _ in range(1000):
    s, _ = simulate_markov_day()
    sim_states.append(s)

sim_states = np.array(sim_states)

markov_same_state = np.mean(
    sim_states[:, 1:]
    ==
    sim_states[:, :-1]
)

observed_same_state = np.mean(
    states[1:] == states[:-1]
)

print(
    "Observed same-state rate :",
    observed_same_state
)

print(
    "Markov simulated rate    :",
    markov_same_state
)

Observed same-state rate : 0.47703079637957535
Markov simulated rate    : 0.4779148936170213


### Conclusion de l'Étape 5

La chaîne de Markov reproduit très précisément la persistance des régimes d'erreur :

$$
47.70\% \text{ observé}
$$

contre :

$$
47.79\% \text{ simulé}
$$

La moyenne des pics change peu, mais les quantiles élevés augmentent.

En particulier :

$$
Q_{0.99}: 88375.8 \rightarrow 88565.7 \text{ MW}
$$

La dépendance entre régimes influence donc davantage la queue de distribution que le comportement moyen.

La chaîne de Markov est conservée comme méthode de simulation de scénarios dépendants.

## Étape 6 — MCMC & Metropolis-Hastings
### Estimation bayésienne des événements extrêmes

### Partie A — Mise en pratique

La méthode POT précédente donnait une seule estimation des paramètres de la loi GPD.

On utilise maintenant Metropolis-Hastings afin d'obtenir une distribution a posteriori des paramètres et des quantiles extrêmes.

In [46]:
from scipy.stats import genpareto

daily_train = (
    y.loc[y.index < "2025-01-01"]
    .resample("D")
    .max()
    .dropna()
)

u = daily_train.quantile(0.95)

excess = (
    daily_train[daily_train > u]
    - u
).values

p_u = len(excess) / len(daily_train)

print("Seuil :", u)
print("Dépassements :", len(excess))

Seuil : 81005.0
Dépassements : 219


In [47]:
rng = np.random.default_rng(42)

def log_posterior(xi, log_beta):

    beta = np.exp(log_beta)

    # prior simple
    if not (-0.8 < xi < 0.8):
        return -np.inf

    lp = (
        -0.5 * ((xi + 0.2) / 0.3) ** 2
        -0.5 * ((log_beta - np.log(4000)) / 1.0) ** 2
    )

    ll = genpareto.logpdf(
        excess,
        c=xi,
        loc=0,
        scale=beta
    )

    if not np.all(np.isfinite(ll)):
        return -np.inf

    return lp + ll.sum()

In [48]:
N = 15000
burn = 3000

samples = np.empty((N, 2))

xi = -0.2
log_beta = np.log(4300)

current_lp = log_posterior(
    xi,
    log_beta
)

accepted = 0

for i in range(N):

    xi_new = xi + rng.normal(0, 0.03)

    log_beta_new = (
        log_beta
        + rng.normal(0, 0.04)
    )

    new_lp = log_posterior(
        xi_new,
        log_beta_new
    )

    if np.log(rng.random()) < (
        new_lp - current_lp
    ):
        xi = xi_new
        log_beta = log_beta_new
        current_lp = new_lp
        accepted += 1

    samples[i] = [
        xi,
        np.exp(log_beta)
    ]

In [49]:
posterior = samples[burn:]

print(
    "Acceptance rate :",
    accepted / N
)

print(
    "xi posterior mean :",
    posterior[:, 0].mean()
)

print(
    "beta posterior mean :",
    posterior[:, 1].mean()
)

Acceptance rate : 0.663
xi posterior mean : -0.1671767247014896
beta posterior mean : 4289.216264477279


In [50]:
def posterior_quantile(
    xi,
    beta,
    p=0.999
):

    if abs(xi) < 1e-8:
        return (
            u
            + beta
            * np.log(
                p_u / (1 - p)
            )
        )

    return (
        u
        + beta / xi
        * (
            (p_u / (1 - p)) ** xi
            - 1
        )
    )


q999_samples = np.array([
    posterior_quantile(xi, beta)
    for xi, beta in posterior
])

In [51]:
print(
    "Q999 posterior mean :",
    q999_samples.mean()
)

print(
    "Q999 95% credible interval :",
    np.quantile(
        q999_samples,
        [0.025, 0.975]
    )
)

Q999 posterior mean : 93329.97272903194
Q999 95% credible interval : [91946.23639288 95307.07292039]


In [52]:
joblib.dump(
    {
        "posterior": posterior,
        "q999_samples": q999_samples,
        "acceptance_rate": accepted / N
    },
    "models/block4_mcmc_evt.joblib"
)

['models/block4_mcmc_evt.joblib']

### Partie C — Analyse

L'estimation classique EVT fournit une valeur ponctuelle de $Q_{0.999}$.

Metropolis-Hastings permet au contraire de propager l'incertitude sur les paramètres :

$$
(\xi,\beta)
$$

jusqu'au quantile extrême.

On obtient ainsi :

$$
P(\xi,\beta \mid données)
$$

puis une distribution de :

$$
Q_{0.999}
$$

au lieu d'une seule valeur.

Cette approche permet donc de quantifier l'incertitude statistique associée aux événements extrêmement rares.

### Conclusion — MCMC et événements extrêmes

Metropolis-Hastings donne des estimations proches de l'approche EVT classique :

$$
E[\xi|data]\approx-0.167
$$

et :

$$
E[\beta|data]\approx4289
$$

Le quantile extrême possède une moyenne a posteriori de :

$$
Q_{0.999}\approx93330\text{ MW}
$$

avec un intervalle crédible à 95 % :

$$
[91946,\ 95307]\text{ MW}
$$

L'estimation ponctuelle obtenue précédemment :

$$
Q_{0.999}\approx93040\text{ MW}
$$

se situe donc bien dans cet intervalle.

Le MCMC permet ainsi de compléter l'estimation EVT en quantifiant l'incertitude sur les événements extrêmement rares.

In [53]:
xi_chain = posterior[:, 0]
beta_chain = posterior[:, 1]

mid = len(posterior) // 2

print(
    "xi mean first/second :",
    xi_chain[:mid].mean(),
    xi_chain[mid:].mean()
)

print(
    "beta mean first/second :",
    beta_chain[:mid].mean(),
    beta_chain[mid:].mean()
)

for lag in [1, 10, 50, 100]:
    print(
        f"xi autocorr lag {lag}:",
        pd.Series(xi_chain).autocorr(lag=lag)
    )

xi mean first/second : -0.16767511184183126 -0.16667833756114792
beta mean first/second : 4287.830060911148 4290.60246804341
xi autocorr lag 1: 0.9464331477448984
xi autocorr lag 10: 0.6467328582144178
xi autocorr lag 50: 0.1157007247129706
xi autocorr lag 100: 0.017027301360071634


### Conclusion du diagnostic MCMC

Les moyennes des deux moitiés de la chaîne sont très proches :

$$
\xi : -0.1677 \approx -0.1667
$$

$$
\beta : 4287.8 \approx 4290.6
$$

La chaîne apparaît donc stable.

L'autocorrélation est élevée à court terme :

$$
\rho_1 \approx 0.95
$$

mais diminue fortement avec le lag :

$$
\rho_{50}\approx0.12
$$

$$
\rho_{100}\approx0.02
$$

La chaîne présente donc une dépendance locale importante mais une convergence globale satisfaisante pour notre application.

# Conclusion du Bloc 4

Ce bloc a permis de passer d'une prévision ponctuelle à la simulation probabiliste de scénarios énergétiques.

Le Monte Carlo indépendant fournit une première distribution de scénarios.

La copule gaussienne permet de conserver la dépendance entre les 48 demi-heures, mais son influence sur le risque de pic reste faible.

La chaîne de Markov reproduit mieux la persistance des régimes d'erreur et augmente notamment le quantile élevé des pics simulés.

Enfin, Metropolis-Hastings permet de propager l'incertitude des paramètres EVT jusqu'au quantile extrême :

$$
Q_{0.999}\approx93330\text{ MW}
$$

avec un intervalle crédible à 95 % :

$$
[91946,\ 95307]\text{ MW}
$$

Le projet combine désormais prévision, incertitude, événements extrêmes et simulation probabiliste dépendante.